In [0]:
# ============================================
# SILVER LAYER PIPELINE
# ============================================

from pyspark.sql.functions import *
from pyspark.sql.types import *

# ============================================
# READ BRONZE DATA
# ============================================

orders_df = spark.read.format("delta").load(
    "/Volumes/workspace/default/lakehouse_raw_data/bronze/orders"
)

customers_df = spark.read.format("delta").load(
    "/Volumes/workspace/default/lakehouse_raw_data/bronze/customers"
)

# ============================================
# CHECK SCHEMA
# ============================================

print("ORDERS SCHEMA")
orders_df.printSchema()

print("CUSTOMERS SCHEMA")
customers_df.printSchema()

# ============================================
# CLEAN COLUMN NAMES
# ============================================

orders_df = orders_df.toDF(
    *[c.strip().lower() for c in orders_df.columns]
)

customers_df = customers_df.toDF(
    *[c.strip().lower() for c in customers_df.columns]
)

# ============================================
# DISPLAY RAW DATA
# ============================================

display(orders_df.limit(20))

# ============================================
# CLEAN DATA
# ============================================

orders_df = orders_df.withColumn(
    "amount",
    regexp_replace(col("amount"), ",", "").cast("double")
)

orders_df = orders_df.withColumn(
    "customer_id",
    col("customer_id").cast("long")
)

orders_df = orders_df.withColumn(
    "order_id",
    col("order_id").cast("long")
)

# ============================================
# FIX DATE FORMAT
# ============================================

orders_df = orders_df.withColumn(
    "order_date",
    to_date(col("order_time"))
)

# ============================================
# REMOVE BAD ROWS
# ============================================

orders_df = orders_df.filter(
    col("amount").isNotNull()
)

orders_df = orders_df.dropDuplicates(["order_id"])

# ============================================
# DEBUG COUNTS
# ============================================

print("Original Orders Count:", spark.read.format("delta").load(
    "/Volumes/workspace/default/lakehouse_raw_data/bronze/orders"
).count())

print("Cleaned Orders Count:", orders_df.count())

print("Null Amounts:", orders_df.filter(
    col("amount").isNull()
).count())

print("Null Status:", orders_df.filter(
    col("status").isNull()
).count())

# ============================================
# DISPLAY STATUS DISTRIBUTION
# ============================================

display(
    orders_df.groupBy("status").count()
)

# ============================================
# DISPLAY CLEAN DATA
# ============================================

display(orders_df.limit(50))

# ============================================
# DELETE OLD SILVER TABLES
# ============================================

dbutils.fs.rm(
    "/Volumes/workspace/default/lakehouse_raw_data/silver/orders_clean",
    recurse=True
)

dbutils.fs.rm(
    "/Volumes/workspace/default/lakehouse_raw_data/silver/customers_clean",
    recurse=True
)

# ============================================
# WRITE SILVER TABLES
# ============================================

orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("order_date") \
    .save(
        "/Volumes/workspace/default/lakehouse_raw_data/silver/orders_clean"
    )

customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        "/Volumes/workspace/default/lakehouse_raw_data/silver/customers_clean"
    )

print("Silver Layer Created Successfully!")

ORDERS SCHEMA
root
 |-- amount: double (nullable = true)
 |-- customer_id: double (nullable = true)
 |-- order_id: long (nullable = true)
 |-- order_time: string (nullable = true)
 |-- status: string (nullable = true)

CUSTOMERS SCHEMA
root
 |-- country: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)



amount,customer_id,order_id,order_time,status
4235.29,6425.0,1,2025-02-26 17:41:00,SUCCESS
-3554.03,5570.0,2,2025-11-29 17:15:00,FAILED
977.88,2643.0,3,2025-11-16 02:05:00,SUCCESS
4298.18,1959.0,4,2025-10-16 17:30:00,SUCCESS
603.99,4695.0,5,2025-09-02 22:49:00,SUCCESS
4365.65,7010.0,6,2025-02-19 14:54:00,SUCCESS
241.2,6543.0,7,2025-06-28 08:29:00,SUCCESS
515.81,9258.0,8,2025-01-28 00:42:00,SUCCESS
2172.64,4874.0,9,2025-03-05 12:42:00,SUCCESS
1975.64,7382.0,10,2025-11-15 21:39:00,SUCCESS


Original Orders Count: 100500
Cleaned Orders Count: 100000
Null Amounts: 0
Null Status: 0


status,count
SUCCESS,85021
PENDING,4932
FAILED,10047


amount,customer_id,order_id,order_time,status,order_date
2352.58,6738,28,2025-05-27 13:07:00,SUCCESS,2025-05-27
4348.67,4877,29,2025-09-22 18:38:00,PENDING,2025-09-22
88.78,6928,30,2025-09-29 12:17:00,SUCCESS,2025-09-29
133.2,2624,33,2025-01-11 15:00:00,SUCCESS,2025-01-11
2707.09,4774,93,2025-07-15 17:40:00,SUCCESS,2025-07-15
1395.15,4081,151,2025-12-18 20:56:00,SUCCESS,2025-12-18
2228.25,3935,171,2025-05-17 02:35:00,SUCCESS,2025-05-17
2762.45,408,200,2025-01-11 20:00:00,SUCCESS,2025-01-11
-3911.73,2248,208,2025-08-03 07:21:00,FAILED,2025-08-03
4673.77,1216,221,2025-01-08 13:43:00,SUCCESS,2025-01-08


Silver Layer Created Successfully!
